# Running Stata in a Jupyter notebook on Google Colab or repo2docker

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/larsvilhuber/jupyter-stata-colab/blob/main/stata_colab_solutions.ipynb)

This notebook installs Stata into the notebook runtime, asks for **your** Stata license (as a base64-encoded string), and then runs Stata code through the `%%stata` cell magic provided by StataCorp's [PyStata](https://www.stata.com/python/pystata19/).

**Before you start**, create the base64-encoded version of your `stata.lic` on your own computer:

```bash
base64 -w0 /usr/local/stata/stata.lic      # Linux
base64 -i /Applications/Stata/stata.lic    # macOS
```

```powershell
[Convert]::ToBase64String([IO.File]::ReadAllBytes("C:\Program Files\Stata19\stata.lic"))   # Windows
```

Copy the resulting single line; you will paste it into a hidden prompt below. It is only written to the runtime's Stata directory, never to the notebook.

## 1. Settings

Stata is copied out of one of the [AEA Data Editor's Stata Docker images](https://hub.docker.com/u/dataeditors) (which contain no license). Set the version using the image-name spelling (e.g. `19_5` for Stata 19.5) and pick a tag, or use `latest` to look up the most recently updated tag on Docker Hub. Pick the edition that matches your license: an MP license can also run SE, so `se` works for both SE and MP licenses.

In [ ]:
import os

VERSION = "19_5"       # image-name spelling, e.g. "19_5" for Stata 19.5
TAG = "2026-08-12"     # a specific tag, or "latest"
EDITION = "se"         # "be", "se" or "mp"
IMAGE = f"dataeditors/stata{VERSION}-{EDITION}:{TAG}"
STATA_ROOT = "/" if os.geteuid() == 0 else os.path.expanduser("~")
STATA_DIR = os.path.join(STATA_ROOT, "usr/local/stata")

## 2. Install Stata

This uses the helper script from this repository (downloading it only if needed) and copies Stata to `/usr/local/stata` on Colab, or `~/usr/local/stata` for non-root users such as repo2docker (about 700 MB, usually well under a minute on Colab).

In [ ]:
if not os.path.exists("setup_stata.py"):
    !wget -q https://raw.githubusercontent.com/larsvilhuber/jupyter-stata-colab/main/setup_stata.py

import setup_stata

setup_stata.install_stata(image=IMAGE, root=STATA_ROOT)

## 3. Enter your license

Paste the base64-encoded content of your `stata.lic` when prompted (the input is hidden). If the environment variable `STATA_LIC_BASE64` is set, it is used instead of prompting.

In [ ]:
from getpass import getpass

license_b64 = os.environ.get("STATA_LIC_BASE64") or getpass("Base64-encoded stata.lic: ")
setup_stata.write_license(license_b64, stata_dir=STATA_DIR)
del license_b64

## 4. Start Stata

This first checks that Stata accepts the license, then initializes PyStata, which registers the `%stata` and `%%stata` magics.

In [ ]:
setup_stata.configure_pystata(stata_dir=STATA_DIR, edition=EDITION)

## 5. Use Stata

Any cell starting with `%%stata` is run by Stata.

In [ ]:
%%stata
sysuse auto, clear
summarize price mpg weight

In [ ]:
%%stata
regress price mpg weight foreign

Graphs are displayed inline:

In [ ]:
%%stata
twoway (scatter price mpg) (lfit price mpg), title("Price vs. mileage")

Data can be moved between Stata and Python (pandas):

In [ ]:
from pystata import stata

df = stata.pdataframe_from_data()
df.groupby("foreign")[["price", "mpg"]].mean()

In [ ]:
%%stata -d df -force
describe, short

## 6. Your turn: a printable table and a figure

The cells below are the solutions to part 4 of the [Reproducible Documents tutorial](https://larsvilhuber.github.io/reproducible-documents/): a clean regression table and a figure, exported to Word and PDF. Run the cells above first.

A clean, printable regression table (`etable`), and a figure that is shown below the cell and saved to disk.

In [ ]:
%%stata
* Estimate three models and store them
sysuse auto, clear
quietly regress price mpg
estimates store m1
quietly regress price mpg weight
estimates store m2
quietly regress price mpg weight i.foreign
estimates store m3

* A clean, printable regression table
etable, estimates(m1 m2 m3) column(index) varlabel  ///
    showstars showstarsnote mstat(N) mstat(r2)      ///
    title("Table 1: Car prices and fuel efficiency")

* A figure: shown below the cell, and saved to disk
twoway (scatter price mpg) (lfit price mpg), by(foreign, note(""))  ///
    ytitle("Price (USD)") xtitle("Mileage (mpg)")
graph export price_mpg.png, replace width(1600)

Put the table and the figure into a Word document (`putdocx`). The number in the sentence comes from Stata, not from you.

In [ ]:
%%stata
* Put the table and the figure into a Word document
estimates restore m2
putdocx clear
putdocx begin
putdocx paragraph, style(Title)
putdocx text ("Car prices and fuel efficiency")
putdocx paragraph
putdocx text ("Holding weight constant, one more mpg changes the price by ")
putdocx text (_b[mpg]), nformat(%5.1f)
putdocx text (" dollars (Table 1, column 2).")
putdocx collect
putdocx paragraph, halign(center)
putdocx image price_mpg.png, width(6)
putdocx save results.docx, replace

The same, as a PDF (`putpdf`).

In [ ]:
%%stata
* The same, as a PDF
estimates restore m2
putpdf clear
putpdf begin
putpdf paragraph, halign(center)
putpdf text ("Car prices and fuel efficiency"), bold
putpdf paragraph
putpdf text ("Holding weight constant, one more mpg changes the price by ")
putpdf text (_b[mpg]), nformat(%5.1f)
putpdf text (" dollars (Table 1, column 2).")
putpdf collect
putpdf paragraph, halign(center)
putpdf image price_mpg.png, width(6)
putpdf save results.pdf, replace

Download both files to your computer (Colab only). Or click the folder icon on the left, right-click a file, and choose *Download*.

In [ ]:
# Download the two files to your computer (Colab only)
from google.colab import files
files.download("results.docx")
files.download("results.pdf")